# SAGE vs PCDFusion (NeurIPS 2026) — cùng 10 cặp FMB + phân tích vùng IR

**Mục tiêu 08/11:** chạy một mô hình IVIF **học sâu** khác (PCDFusion) trên **đúng** 10 ID đã chạy SAGE, tái tính các metric bằng **cùng hàm và cùng nguồn ảnh**; kiểm tra vùng có nhiệt nổi bật nhưng ít thông tin Visible để tìm *ứng viên failure cases*. 

- SAGE: **dùng lại ảnh fused đã có** trong Kaggle Input chứa `fused_png/`, **không chạy lại pretrained SAGE Student**.
- PCDFusion: chạy pretrained checkpoint chính thức tại `BIT-RuiLiu/PCDFusion`, pin mã commit và SHA256 checkpoint.
- Đối chứng bổ sung: `Y_AVG` / `Y_MAX` (không được gọi là SOTA).
- Output: ZIP kết quả gồm 40 bộ metric (4 phương pháp × 10 ID), bảng chênh lệch từng ảnh, mask vùng IR theo quy tắc khóa trước, hình so sánh và biên bản tự sinh.

**Quan trọng:** Mọi metric ở đây là bản tính độc lập bằng công thức đã ghi; `SCD`, `MS-SSIM` **chưa được xác nhận tương thích tuyệt đối** công thức và cấu hình báo cáo của paper. Một vùng sáng IR không đồng nghĩa có người/vật thể được gắn nhãn. Không điều chỉnh ngưỡng trên FMB/test.

Tài liệu: https://arxiv.org/abs/2610.23663 · Code: https://github.com/BIT-RuiLiu/PCDFusion

## Cell 1 — Khởi tạo và cố định 10 image IDs

In [ ]:
from pathlib import Path
import os, sys, json, io, tarfile, zipfile, csv, hashlib, subprocess, platform, time, shutil
from datetime import datetime, timezone
import numpy as np
import cv2
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

IDS = ['00014','00024','00061','00064','00071','00075','00104','00109','00112','00145']
assert len(IDS)==len(set(IDS))==10
BASE=Path('/kaggle/working')
INPUT=Path('/kaggle/input')
assert BASE.is_dir(), 'Hãy chạy trong Kaggle Notebook.'
RUN_ROOT=BASE/('sage_pcdfusion_08nov_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
RUN_ROOT.mkdir(parents=True,exist_ok=False)
for sub in ['metrics','figures','logs','fused/PCDFusion','dataset/FMB/test/ir','dataset/FMB/test/vi']:
    (RUN_ROOT/sub).mkdir(parents=True,exist_ok=True)
print('RUN_ROOT:',RUN_ROOT)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('IDs:', ', '.join(IDS))

## Cell 2 — Tìm TAR FMB/test (hoặc tải đúng nguồn cũ) và kiểm tra cặp nguồn

In [ ]:
# KHÔNG dùng ảnh demo khác; không resize hoặc thay split.
TARS=sorted(BASE.rglob('fmb-test-00000.tar'))+sorted(INPUT.rglob('fmb-test-00000.tar'))
if not TARS:
    print('Không có TAR trong phiên Kaggle này. Đang tải đúng bản FMB/test dùng trước đó...')
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub'],check=True)
        from huggingface_hub import hf_hub_download
    try:
        p=hf_hub_download(repo_id='LC-State/RGB2IR_Translation',repo_type='dataset',
            filename='data/FMB/test/fmb-test-00000.tar',local_dir=str(BASE/'sage_hf_cache'))
        TARS=[Path(p)]
    except Exception as e:
        raise RuntimeError('Không tải được TAR; bật Internet hoặc Add Input TAR FMB. '+str(e)) from e
TAR_FILE=TARS[0]
print('TAR:',TAR_FILE)

def decode(b,mode):
    with Image.open(io.BytesIO(b)) as im:
        return np.asarray(im.convert(mode)).copy()
SOURCES={}
with tarfile.open(TAR_FILE,'r:*') as tf:
    members={m.name.rsplit('/',1)[-1]:m for m in tf if m.isfile()}
    for sid in IDS:
        assert f'{sid}.rgb.png' in members and f'{sid}.ir.png' in members, f'TAR thiếu ID {sid}'
        vb=tf.extractfile(members[f'{sid}.rgb.png']).read()
        ib=tf.extractfile(members[f'{sid}.ir.png']).read()
        vis=decode(vb,'RGB'); ir=decode(ib,'L')
        assert vis.shape[:2]==ir.shape, f'Mismatch IR/VIS: {sid}'
        SOURCES[sid]={'vis':vis,'ir':ir, 'vb':vb,'ib':ib,
            'vis_sha256':hashlib.sha256(vb).hexdigest(),'ir_sha256':hashlib.sha256(ib).hexdigest()}
assert len(SOURCES)==10
print('Số cặp:',len(SOURCES),'— kích thước mẫu H,W:',SOURCES[IDS[0]]['ir'].shape)

## Cell 3 — Đọc đúng 10 fused PNG SAGE đã gắn bằng Add Input

In [ ]:
folders=sorted([p for p in list(INPUT.rglob('fused_png'))+list(BASE.rglob('fused_png')) if p.is_dir()],key=str)
SAGE={}; SAGE_SRC=None
for folder in folders:
    if all((folder/f'{sid}.png').is_file() for sid in IDS):
        SAGE={sid:decode((folder/f'{sid}.png').read_bytes(),'RGB') for sid in IDS}
        SAGE_SRC=str(folder)
        break
if not SAGE:
    zips=list(INPUT.rglob('SAGE_LocBe_pretrained_inference_results.zip'))+list(BASE.rglob('SAGE_LocBe_pretrained_inference_results.zip'))
    for zp in zips:
        with zipfile.ZipFile(zp) as z:
            files={n.rsplit('/',1)[-1]:n for n in z.namelist() if '/fused_png/' in '/'+n}
            if all(f'{sid}.png' in files for sid in IDS):
                SAGE={sid:decode(z.read(files[f'{sid}.png']),'RGB') for sid in IDS}
                SAGE_SRC=str(zp)
                break
if not SAGE:
    print('Các fused_png tìm thấy:',[str(x) for x in folders])
    raise FileNotFoundError('Chưa có đủ 10 fused SAGE. Kaggle Add Input bộ output SAGE chứa fused_png.')
for sid in IDS:
    assert SAGE[sid].shape==SOURCES[sid]['vis'].shape, f'SAGE không khớp kích thước: {sid}'
print('Nguồn SAGE:',SAGE_SRC)
print('Đã xác minh 10/10 PNG RGB cùng ID và shape với dữ liệu FMB.')

## Cell 4 — Cài thư viện và tải PCDFusion pretrained theo commit cố định

Checkpoint không phải của SAGE. Đoạn này chỉ tải PCDFusion và kiểm tra toàn vẹn; **không fine-tune**.

In [ ]:
PCDF_REPO='https://github.com/BIT-RuiLiu/PCDFusion.git'
PCDF_COMMIT='4285bdca266607a29a560904faa37425e8014e6c'
PCDF_CKPT_SHA='1c105152cbd31c142867e1ef97035e8fb1a4d4619b7ac0043166f4b05d4a0a61'
PCDF_DIR=BASE/'PCDFusion_4285'
if not PCDF_DIR.exists():
    subprocess.run(['git','clone',PCDF_REPO,str(PCDF_DIR)],check=True)
subprocess.run(['git','-C',str(PCDF_DIR),'checkout','--detach',PCDF_COMMIT],check=True)
repo_head=subprocess.check_output(['git','-C',str(PCDF_DIR),'rev-parse','HEAD'],text=True).strip()
assert repo_head==PCDF_COMMIT, 'Git commit sai: '+repo_head
ckpt=PCDF_DIR/'weights'/'best.pth'
assert ckpt.is_file() and ckpt.stat().st_size>1_000_000, 'PCDFusion pretrained checkpoint không tồn tại / là Git LFS pointer.'
actual_ckpt_sha=hashlib.sha256(ckpt.read_bytes()).hexdigest()
assert actual_ckpt_sha==PCDF_CKPT_SHA, 'SHA-256 checkpoint khác bản đã kiểm chứng: '+actual_ckpt_sha
print('PCDFusion Git SHA:',repo_head)
print('Checkpoint SHA-256:',actual_ckpt_sha, 'size MB:', round(ckpt.stat().st_size/1e6,2))
# Dùng môi trường PyTorch/Kaggle đang có; không ép hạ Torch/CUDA.
subprocess.run([sys.executable,'-m','pip','install','-q','pytorch-msssim','PyYAML','tqdm'],check=True)

## Cell 5 — Tạo cùng 10 cặp PNG cho PCDFusion; kiểm tra hash giữ nguyên dữ liệu

In [ ]:
PCDF_DATA=RUN_ROOT/'dataset'/'FMB'
for sid in IDS:
    irp=PCDF_DATA/'test'/'ir'/f'{sid}.png'
    vip=PCDF_DATA/'test'/'vi'/f'{sid}.png'
    irp.write_bytes(SOURCES[sid]['ib'])
    vip.write_bytes(SOURCES[sid]['vb'])
    assert hashlib.sha256(irp.read_bytes()).hexdigest()==SOURCES[sid]['ir_sha256']
    assert hashlib.sha256(vip.read_bytes()).hexdigest()==SOURCES[sid]['vis_sha256']
print('Đã chuẩn bị 10/10 cặp: ',PCDF_DATA/'test')

## Cell 6 — Chạy pretrained PCDFusion trên đúng 10 cặp FMB

Thời gian ghi nhận gồm cả khởi tạo Python/load model; **không so trực tiếp** với forward 21,141 ms của SAGE.

In [ ]:
PCDF_OUTPUT=RUN_ROOT/'fused'/'PCDFusion'
cmd=[sys.executable,str(PCDF_DIR/'infer.py'),
     '--root_dir',str(PCDF_DATA),'--phase','test',
     '--ckpt',str(ckpt),'--out_dir',str(PCDF_OUTPUT),
     '--device','cuda' if torch.cuda.is_available() else 'cpu',
     '--pad_multiple','32','--num_workers','0']
print('Chạy PCDFusion:', ' '.join(cmd))
t0=time.perf_counter()
res=subprocess.run(cmd,cwd=str(PCDF_DIR),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
wall_sec=time.perf_counter()-t0
(RUN_ROOT/'logs'/'pcdfusion_infer.log').write_text(res.stdout,encoding='utf-8')
if res.returncode!=0:
    print(res.stdout[-7000:])
    raise RuntimeError('PCDFusion inference thất bại; xem logs/pcdfusion_infer.log (mã lỗi '+str(res.returncode)+')')
print('PCDFusion hoàn tất, tổng wall-clock (gồm load):',round(wall_sec,2),'giây')
print('\n'.join(res.stdout.splitlines()[-12:]))
PCDF={}
for sid in IDS:
    p=PCDF_OUTPUT/f'{sid}.png'
    assert p.is_file(),f'PCDFusion thiếu {p}'
    img=np.array(Image.open(p).convert('RGB'))
    assert img.shape==SOURCES[sid]['vis'].shape, f'Kích thước PCDF không khớp {sid}'
    PCDF[sid]=img
assert len(PCDF)==10
print('Xác nhận 10/10 PCDFusion PNG RGB cùng kích thước')

## Cell 7 — Tạo hai đối chứng cổ điển và khóa danh sách phương pháp

In [ ]:
def visible_y(vis):
    return cv2.cvtColor(vis,cv2.COLOR_RGB2YCrCb)[:,:,0]
def y_to_rgb(vis,y):
    ycc=cv2.cvtColor(vis,cv2.COLOR_RGB2YCrCb).copy()
    ycc[:,:,0]=np.clip(np.rint(y),0,255).astype(np.uint8)
    return cv2.cvtColor(ycc,cv2.COLOR_YCrCb2RGB)
OUTPUTS={'SAGE':SAGE,'PCDFusion':PCDF,'Y_AVG':{},'Y_MAX':{}}
for sid in IDS:
    vi=SOURCES[sid]['vis']; ir=SOURCES[sid]['ir'].astype(np.float64)
    y=visible_y(vi).astype(np.float64)
    OUTPUTS['Y_AVG'][sid]=y_to_rgb(vi,(y+ir)/2)
    OUTPUTS['Y_MAX'][sid]=y_to_rgb(vi,np.maximum(y,ir))
    for name,imgs in OUTPUTS.items():
        assert imgs[sid].shape==vi.shape, (name,sid)
print('Đối chứng:',list(OUTPUTS),' × ',len(IDS),'ảnh = ',len(OUTPUTS)*len(IDS),'dòng metric')

## Cell 8 — Các hàm metric và vùng IR nổi bật

- **EN, SD:** ảnh fused RGB chuyển xám 8-bit, histogram 256 bins, SD dân số.
- **SCD:** `corr(F − IR, VIS_Y) + corr(F − VIS_Y, IR)` sau cast float64.
- **MS-SSIM tổng:** `MS_SSIM(F, IR) + MS_SSIM(F, VIS_Y)`, cùng backend/cấu hình cho mọi method.
- **IR-only candidate mask (thăm dò):** IR ≥ P85, VIS_Y ≤ P50, và IR−VIS_Y ≥ 20/255; mọi ngưỡng được **khóa trước, không tune theo kết quả test**.
- **Local contrast:** chênh lệch mean vùng mask và một vành lân cận 7 px. `local_IR_retention` = contrast_fused / contrast_IR nếu contrast IR > 5. `local_IR_abs_error` = |contrast_fused−contrast_IR|, không giả định càng sáng càng tốt.
- Bỏ qua ảnh có mask quá nhỏ, vành không hợp lệ hoặc contrast IR không đủ. Những phép đo vùng chỉ là **proxy theo pixel**, không phải target recall/mIoU.

In [ ]:
from pytorch_msssim import ms_ssim
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
def gray(rgb): return cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
def EN(g):
    h=np.bincount(g.reshape(-1),minlength=256).astype(np.float64)
    p=h[h>0]/h.sum()
    return float(-(p*np.log2(p)).sum())
def pearson(a,b):
    a=np.asarray(a,dtype=np.float64).ravel(); b=np.asarray(b,dtype=np.float64).ravel()
    a=a-a.mean(); b=b-b.mean()
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den>1e-12 else float('nan')
def SCD(f,ir,v):
    f=np.asarray(f,dtype=np.float64); ir=np.asarray(ir,dtype=np.float64);v=np.asarray(v,dtype=np.float64)
    return pearson(f-ir,v)+pearson(f-v,ir)
def MSSIM(x,y):
    xx=torch.from_numpy(np.ascontiguousarray(x,dtype=np.float32))[None,None].to(DEVICE)/255
    yy=torch.from_numpy(np.ascontiguousarray(y,dtype=np.float32))[None,None].to(DEVICE)/255
    with torch.inference_mode():
        return float(ms_ssim(xx,yy,data_range=1.0,size_average=True).item())
def sobelmag(x):
    x=np.asarray(x,dtype=np.float64)
    return np.hypot(cv2.Sobel(x,cv2.CV_64F,1,0,ksize=3),cv2.Sobel(x,cv2.CV_64F,0,1,ksize=3))
def candidate_roi(ir,vi):
    ir=ir.astype(np.float64); vi=vi.astype(np.float64)
    return (ir>=np.quantile(ir,0.85))&(vi<=np.quantile(vi,0.50))&((ir-vi)>=20)
def roi_diagnostic(f,ir,vi):
    mask=candidate_roi(ir,vi)
    coverage=float(mask.mean())
    if int(mask.sum())<64:
        return {'coverage':coverage,'roi_n':int(mask.sum()),'local_ir_contrast':np.nan,
                'local_f_contrast':np.nan,'local_IR_retention':np.nan,'local_IR_abs_error':np.nan}
    dil=cv2.dilate(mask.astype('uint8'),np.ones((15,15),np.uint8))>0
    ring=dil&(~mask)
    if int(ring.sum())<64:
        return {'coverage':coverage,'roi_n':int(mask.sum()),'local_ir_contrast':np.nan,
                'local_f_contrast':np.nan,'local_IR_retention':np.nan,'local_IR_abs_error':np.nan}
    ir=np.asarray(ir,dtype=np.float64);f=np.asarray(f,dtype=np.float64)
    c_ir=float(ir[mask].mean()-ir[ring].mean());c_f=float(f[mask].mean()-f[ring].mean())
    ok=c_ir>5.0
    return {'coverage':coverage,'roi_n':int(mask.sum()),'local_ir_contrast':c_ir,
            'local_f_contrast':c_f,'local_IR_retention':(c_f/c_ir) if ok else np.nan,
            'local_IR_abs_error':abs(c_f-c_ir) if ok else np.nan}
# kiểm tra sanity: xác nhận phép trừ float tránh uint8 wrap
rng=np.random.default_rng(42); aa=rng.integers(0,256,(32,32),dtype=np.uint8);bb=rng.integers(0,256,(32,32),dtype=np.uint8)
ff=(aa.astype(float)+bb.astype(float))/2
assert np.isfinite(SCD(ff,aa,bb))
assert np.isclose(SCD(ff,aa,bb),SCD(ff+4,aa,bb))
assert candidate_roi(aa,bb).shape==aa.shape
print('Kiểm tra sơ bộ metric và mask PASS. Backend MS-SSIM:',DEVICE)

## Cell 9 — Đánh giá toàn bộ phương pháp trên cùng ảnh nguồn, export CSV

In [ ]:
rows=[]
for sid in IDS:
    ir=SOURCES[sid]['ir']; v=visible_y(SOURCES[sid]['vis'])
    for method,images in OUTPUTS.items():
        f=gray(images[sid]); diag=roi_diagnostic(f,ir,v)
        mi=MSSIM(f,ir); mv=MSSIM(f,v)
        rows.append({'image_id':sid,'method':method,'EN':EN(f),
            'SD':float(np.std(f.astype(np.float64),ddof=0)),
            'SCD':SCD(f,ir,v),'MS_SSIM_IR':mi,'MS_SSIM_VIS':mv,
            'MS_SSIM_sum':mi+mv,
            'edge_corr_IR':pearson(sobelmag(f),sobelmag(ir)),
            'edge_corr_VIS':pearson(sobelmag(f),sobelmag(v)),
            **diag,'valid':True,'error_reason':''})
per=pd.DataFrame(rows)
assert len(per)==40 and (per.groupby('image_id').method.nunique()==4).all()
per.to_csv(RUN_ROOT/'metrics'/'per_image.csv',index=False,float_format='%.9f')
fields=['EN','SD','SCD','MS_SSIM_sum','edge_corr_IR','edge_corr_VIS','local_IR_retention','local_IR_abs_error']
summary=per.groupby('method')[fields].agg(['mean','std','median','count'])
summary.to_csv(RUN_ROOT/'metrics'/'summary.csv',float_format='%.9f')
print('40/40 metric rows; ROI hợp lệ:',int(per[per.method=='SAGE'].local_IR_retention.notna().sum()),'/10')
display(summary.round(4))

## Cell 10 — So sánh PCDFusion trừ SAGE trên từng ID và phân tích sự đánh đổi

In [ ]:
paired=[]
for sid in IDS:
    a=per[(per.image_id==sid)&(per.method=='SAGE')].iloc[0]
    b=per[(per.image_id==sid)&(per.method=='PCDFusion')].iloc[0]
    rec={'image_id':sid,'comparison':'PCDFusion_minus_SAGE'}
    for key in fields:
        rec['SAGE_'+key]=float(a[key]); rec['PCDFusion_'+key]=float(b[key]);
        rec['delta_'+key]=float(b[key]-a[key])
    paired.append(rec)
paired=pd.DataFrame(paired)
paired.to_csv(RUN_ROOT/'metrics'/'paired_comparison.csv',index=False,float_format='%.9f')
print('Số ảnh có GIÁ TRỊ PCDFusion > SAGE (n hợp lệ; local_IR_abs_error càng thấp càng tốt):')
for f in fields:
    delta=paired['delta_'+f].dropna()
    print(f'  {f}: {(delta>0).sum()}/{len(delta)}, median delta={delta.median():+.4f}')
display(paired[['image_id','delta_SCD','delta_MS_SSIM_sum','delta_edge_corr_IR','delta_local_IR_retention','delta_local_IR_abs_error']].round(4))

## Cell 11 — Vẽ mask proxy và ảnh so sánh (không gán nhãn failure case nếu chưa được xác nhận)

In [ ]:
# Chọn ID có local_IR_retention SAGE thấp nhất trong các ROI hợp lệ; KHÔNG dùng để điều chỉnh mô hình.
sage_roi=per[per.method=='SAGE'].sort_values('local_IR_retention',na_position='last')
selected=sage_roi.image_id.head(3).tolist()
fig,axs=plt.subplots(len(selected),6,figsize=(18,3.0*len(selected)),squeeze=False)
for i,sid in enumerate(selected):
    ir=SOURCES[sid]['ir'];vis=SOURCES[sid]['vis']; v=visible_y(vis)
    mask=candidate_roi(ir,v)
    overlay=vis.copy();overlay[mask]=(overlay[mask].astype(np.float64)*0.45+np.array([255,40,40])*0.55).clip(0,255).astype(np.uint8)
    panels=[ir,vis,overlay,SAGE[sid],PCDF[sid],OUTPUTS['Y_AVG'][sid]]
    names=['IR','Visible','IR-only proxy mask','SAGE','PCDFusion','Y_AVG']
    for j,(img,name) in enumerate(zip(panels,names)):
        axs[i,j].imshow(img,cmap='gray' if j==0 else None)
        axs[i,j].set_title(f'{sid} • {name}',fontsize=10)
        axs[i,j].axis('off')
plt.tight_layout()
fig.savefig(RUN_ROOT/'figures'/'region_candidates_3ids.png',dpi=130,bbox_inches='tight')
plt.show()
# thêm toàn bộ 10 mask để có thể kiểm chứng không chỉ các ảnh chọn lọc
for sid in IDS:
    mask=candidate_roi(SOURCES[sid]['ir'],visible_y(SOURCES[sid]['vis']))
    Image.fromarray((mask*255).astype(np.uint8)).save(RUN_ROOT/'figures'/f'mask_{sid}.png')

## Cell 12 — Manifest, provenance, biên bản kết quả tự sinh và ZIP bàn giao

In [ ]:
manifest=[]
for sid in IDS:
    s=SOURCES[sid]
    manifest.append({'image_id':sid,'split':'FMB/test_repackaged',
        'source_tar':TAR_FILE.name,'ir_member':f'{sid}.ir.png','vis_member':f'{sid}.rgb.png',
        'height':s['ir'].shape[0],'width':s['ir'].shape[1],
        'ir_sha256':s['ir_sha256'],'vis_sha256':s['vis_sha256'],
        'sage_rgb_pixels_sha256':hashlib.sha256(SAGE[sid].tobytes()).hexdigest(),
        'pcdfusion_sha256':hashlib.sha256((PCDF_OUTPUT/f'{sid}.png').read_bytes()).hexdigest()})
pd.DataFrame(manifest).to_csv(RUN_ROOT/'manifest.csv',index=False)
run_info={'run_id':RUN_ROOT.name,'created_utc':datetime.now(timezone.utc).isoformat(),
    'purpose':'Frozen pretrained SAGE versus PCDFusion; 10 identical FMB/test-repackaged image IDs',
    'ids':IDS,'dataset_tar':str(TAR_FILE),'sage_source':SAGE_SRC,
    'pcdfusion_repository':PCDF_REPO,'pcdfusion_commit':repo_head,
    'pcdfusion_weights_sha256':actual_ckpt_sha,'pcdfusion_infer_args':cmd,
    'pcdfusion_wall_clock_seconds_includes_loading':round(wall_sec,6),
    'measurements':{
       'EN':'RGB2GRAY 8-bit Shannon entropy, 256 levels',
       'SD':'RGB2GRAY population std in pixel units 0-255',
       'SCD':'corr(F-IR,VIS_Y)+corr(F-VIS_Y,IR) float64',
       'MS_SSIM_sum':'pytorch_msssim.ms_ssim(F,IR)+ms_ssim(F,VIS_Y), [0,1]',
       'region_mask':'IR>=image P85 AND VIS_Y<=image P50 AND IR-VIS_Y>=20 (uint8 0..255)',
       'local_IR_retention':'(mean(F,ROI)-mean(F,ring))/(mean(IR,ROI)-mean(IR,ring)); ring dilation 15x15; source contrast>5',
       'local_IR_abs_error':'abs(fused local contrast - infrared local contrast), lower is better'},
    'cautions':['Results are 10 image exploratory results, not official FMB full test.',
       'Region mask is pixel heuristic, not person/semantic ground truth.',
       'MS-SSIM/SCD not proven exactly parity with authors official protocol.',
       'Test IDs were used previously for exploratory analysis; do not tune proposals on test.',
       'Total PCDFusion subprocess time is NOT comparable to SAGE forward latency.',
       'Paper reported results never substituted for measured 10-image scores.']}
(RUN_ROOT/'run_config.json').write_text(json.dumps(run_info,ensure_ascii=False,indent=2),encoding='utf-8')
(RUN_ROOT/'environment.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,
    'torchvision':__import__('torchvision').__version__,'numpy':np.__version__,'opencv':cv2.__version__,
    'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'platform':platform.platform()},ensure_ascii=False,indent=2),encoding='utf-8')
sum_means=per.groupby('method')[['EN','SD','SCD','MS_SSIM_sum','edge_corr_IR','edge_corr_VIS','local_IR_retention']].mean()
pivot=sum_means.round(4).to_markdown()
paired_mean=paired[[f'delta_{key}' for key in ['SCD','MS_SSIM_sum','edge_corr_IR','local_IR_retention']]].mean().round(4)
md_report=f"""# Báo cáo thí nghiệm SAGE vs PCDFusion — ngày 08/11/2026

## Tóm tắt thiết lập
- SAGE pretrained Student: giữ nguyên 10 PNG cũ, cùng FMB/test repackaged.
- PCDFusion: pretrained checkpoint commit `{repo_head}`, SHA256 `{actual_ckpt_sha}`.
- 10 image IDs: {', '.join(IDS)}. Không fine-tune phương pháp nào.
- PCDFusion subprocess wall-clock gồm load/checkpoint: {wall_sec:.2f} giây.

## Bảng mean trên 10 ảnh (kết quả TỰ CHẠY notebook này)
{pivot}

## Paired delta PCDFusion - SAGE (trung bình)
{paired_mean.to_string()}

## Đọc kết quả đúng cách
- Metric trung bình không cho phép suy ra target detection/semantic segmentation.
- Chỉ xem ``local_IR_retention`` như proxy, không phải IR-target recall.
- Nếu phương pháp khác không có ưu thế nhất quán, hãy ghi 'chưa thấy bằng chứng cải thiện nhất quán trên 10 ảnh' chứ không kết luận tổng quát.
- Xem ``metrics/per_image.csv`` và ``metrics/paired_comparison.csv`` cùng hình/mask các ID.
- Chưa so sánh mIoU; cần segmentation evaluator + ground-truth label trên cùng split.
- 10 ảnh là phép khảo sát bước đầu; để lựa chọn phương pháp mới phải có dev từ train độc lập.

Nguồn PCDFusion: https://arxiv.org/abs/2610.23663; mã: https://github.com/BIT-RuiLiu/PCDFusion.
"""
(RUN_ROOT/'BAO_CAO_TU_DONG.md').write_text(md_report,encoding='utf-8')
(RUN_ROOT/'README.md').write_text('Verified real-run artifacts: run_config, environment, manifest, metrics CSV, outputs PCDFusion, ROI candidate masks and source comparison images. SAGE source reference in config. No new SAGE inference; this run is PCDFusion vs SAGE on the same IDs.\n',encoding='utf-8')
print(md_report)
archive=shutil.make_archive(str(RUN_ROOT),'zip',root_dir=RUN_ROOT)
print('\nZIP kết quả để gửi lại:',archive)

## Trước khi báo cáo

1. Mở `BAO_CAO_TU_DONG.md`, kiểm tra các số liệu được tính thật từ `per_image.csv`.
2. Dùng hình `region_candidates_3ids.png` để nêu **ứng viên** thất bại, không suy ra mất người chỉ từ proxy.
3. So sánh phương pháp hiện đại PCDFusion với SAGE theo **paired delta** trên cùng 10 ID; `Y_AVG/Y_MAX` chỉ giúp đối chiếu kỹ thuật cơ bản.
4. Nếu cần khẳng định novelty, bắt buộc đánh giá thêm tập dev/training, downstream segmentation và nhiều phương pháp mạnh hơn.
5. Chia sẻ ZIP output Kaggle, không push ảnh FMB thô hoặc checkpoint lên GitHub nhóm.